All this file does it ingest the raw csv files and converts them to delta tables adding them to the appropriate catalogs

The catalogs are also set up for current and future use

In [0]:
%sql
-- First check what catalogs you have
SHOW CATALOGS;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS unity_catalog
MANAGED LOCATION 'abfss://unity-catalog-storage@<storage-account>.dfs.core.windows.net/';

-- Then create schemas inside
CREATE SCHEMA IF NOT EXISTS unity_catalog.bronze;
CREATE SCHEMA IF NOT EXISTS unity_catalog.silver;
CREATE SCHEMA IF NOT EXISTS unity_catalog.gold;

In [0]:
%sql
DROP CATALOG IF EXISTS unity_catalog CASCADE;

In [0]:
%sql
-- Create a dev catalog
CREATE CATALOG IF NOT EXISTS dev
MANAGED LOCATION 'abfss://unity-catalog-storage@<storage-account>.dfs.core.windows.net/dev/';

-- Then create schemas inside
CREATE SCHEMA IF NOT EXISTS dev.bronze;
CREATE SCHEMA IF NOT EXISTS dev.silver;
CREATE SCHEMA IF NOT EXISTS dev.gold;



In [0]:
%sql
-- Create a dev catalog
CREATE CATALOG IF NOT EXISTS staging
MANAGED LOCATION 'abfss://unity-catalog-storage@<storage-account>.dfs.core.windows.net/staging/';

-- Then create schemas inside
CREATE SCHEMA IF NOT EXISTS staging.bronze;
CREATE SCHEMA IF NOT EXISTS staging.silver;
CREATE SCHEMA IF NOT EXISTS staging.gold;



In [0]:
%sql
-- Create a prod catalog
CREATE CATALOG IF NOT EXISTS prod
MANAGED LOCATION 'abfss://unity-catalog-storage@<storage-account>.dfs.core.windows.net/prod/';

-- Then create schemas inside
CREATE SCHEMA IF NOT EXISTS prod.bronze;
CREATE SCHEMA IF NOT EXISTS prod.silver;
CREATE SCHEMA IF NOT EXISTS prod.gold;



In [0]:
files = dbutils.fs.ls("abfss://raw-data@<storage-account>.dfs.core.windows.net/")

for file in files:
    table_name = file.name.replace(".csv", "").replace("-", "_").lower()
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file.path)
    
    # Sanitize column names by replacing invalid characters with underscores
    for col in df.columns:
        df = df.withColumnRenamed(col, col.replace(" ", "_").replace(",", "_").replace(";", "_").replace("{", "_").replace("}", "_").replace("(", "_").replace(")", "_").replace("\n", "_").replace("\t", "_").replace("=", "_"))
    
    df.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://unity-catalog-storage@<storage-account>.dfs.core.windows.net/dev/bronze/{table_name}")
    print(f"Written: {table_name}")

Below we are reading in the other dataset which will be combined with my data above to create one dataset

In [0]:
# Get list of tables from samples.accuweather schema
tables = spark.sql("SHOW TABLES IN samples.accuweather").collect()

for table in tables:
    table_name = table.tableName
    
    # Skip internal tables
    if table_name.startswith("_"):
        continue
    
    # Read the table from samples.accuweather
    df = spark.table(f"samples.accuweather.{table_name}")
    
    # Sanitize column names
    for col in df.columns:
        df = df.withColumnRenamed(col, col.replace(" ", "_").replace(",", "_").replace(";", "_").replace("{", "_").replace("}", "_").replace("(", "_").replace(")", "_").replace("\n", "_").replace("\t", "_").replace("=", "_"))
    
    # Write to dev bronze layer
    df.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://unity-catalog-storage@<storage-account>.dfs.core.windows.net/dev/bronze/{table_name}")
    print(f"Written: {table_name}")